# **Data Collection and Processing**

**Team:** David Jorge-Bates and Jonah Fishman
**Course:** CIS 2450 - Big Data Analytics, Spring 2026
**Professor:** Zachary G. Ives
**Submission:** April 30, 2026

> ## ⚠️ Warning before running this notebook
>
> This notebook documents how `data/song_data.db` was originally built — extracting from Kaggle's 550K+ Spotify Songs CSV, enriching via the Deezer API, and joining the two sources.
>
> **The Deezer API requests in Step 2c takes several days** to run across all 498,978 songs due to request limits and network speeds. The prebuilt database used for analysis is provided as `data/song_data.db` in our zipped codebase, so graders should not use this notebook to build a data file for grading purposes. It exists only as a record of the ETL pipeline and for demonstration purposes to show how the database was made.
>
> To run the analysis on the full dataset, open `genre_prediction.ipynb` instead. To launch the dashboard, run `python dashboard.py` from the project root.
>
> ## ⚠️ Warning about `songs.csv`
>
> Our GitHub repo does not include songs.csv due to its large file size. Please see the zipped codebase for this file.

## **Step 0:** Imports & Setup

In [1]:
import duckdb
from datetime import datetime
import requests

In [2]:
songsCSV = 'songs.csv'
SONGS_DB = 'songs.db'
con = duckdb.connect(SONGS_DB)

## **Step 1:** Extracting and cleaning data from Spotify dataset

### **Step 1a:** Pulling in raw data from songs.csv

In [3]:
con.execute('DROP TABLE IF EXISTS songs_raw')
con.execute(f"""
    CREATE TABLE songs_raw
    AS SELECT *
    FROM read_csv('{songsCSV}')
""")
print('Num rows:', con.execute(
    """SELECT COUNT(*)
    FROM songs_raw"""
    ).fetchall()
)

Num rows: [(550622,)]


In [4]:
con.table('songs_raw').limit(5).show()

┌────────────────────────┬─────────────────────┬─────────────────────┬────────────────────┬────────────────────┬────────┬───────┬──────────┬───────┬─────────────┬──────────────┬────────────────────────┬──────────┬────────────────────┬─────────┬─────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

### **Step 1b:** Remove duplicates from songs_raw table

In many cases in our songs_raw table, there exists multiple copies of the same song by the same artist but released across multiple albums (e.g. single and full album releases). Below is an example:

In [5]:
con.sql("""
SELECT id, name, artists, album_name
FROM songs_raw
WHERE name LIKE 'A Foggy Day'
AND artists LIKE '%Frank Sinatra%'
""").show()

┌────────────────────────┬─────────────┬───────────────────┬──────────────────────────────────────────────┐
│           id           │    name     │      artists      │                  album_name                  │
│        varchar         │   varchar   │      varchar      │                   varchar                    │
├────────────────────────┼─────────────┼───────────────────┼──────────────────────────────────────────────┤
│ 23B4EhyBqrhpvBFGk7MU7l │ A Foggy Day │ ["Frank Sinatra"] │ Sinatra Sings Gershwin                       │
│ 55MaJtfKTBDHRf2pu9Lmvu │ A Foggy Day │ ["Frank Sinatra"] │ Ring-A-Ding-Ding! (50th Anniversary Edition) │
│ 2CVjnFQ19WH4r9BuTTEWGH │ A Foggy Day │ ["Frank Sinatra"] │ Songs For Young Lovers                       │
└────────────────────────┴─────────────┴───────────────────┴──────────────────────────────────────────────┘



The solution:

In [6]:
con.execute('DROP TABLE IF EXISTS songs_no_duplicates')
con.execute("""
    CREATE TABLE songs_no_duplicates (
        _id VARCHAR,
        song_name VARCHAR,
        artists_list VARCHAR,
        duration_ms BIGINT,
        lyrics VARCHAR,
        genre VARCHAR,
        niche_genres VARCHAR,
        loudness DOUBLE,
        speechiness DOUBLE,
        instrumentalness DOUBLE,
        liveness DOUBLE,
        valence DOUBLE,
        tempo DOUBLE,
        key BIGINT,
        mode BIGINT,
        acousticness DOUBLE,
        danceability DOUBLE,
        energy DOUBLE,
        popularity BIGINT,
        PRIMARY KEY(_id)
    )
""")

In [7]:
con.sql("""
        INSERT INTO songs_no_duplicates (
             _id,
             song_name,
             artists_list,
             duration_ms,
             lyrics,
             genre,
             niche_genres,
             loudness,
             speechiness,
             instrumentalness,
             liveness,
             valence,
             tempo, 
             key,
             mode,
             acousticness,
             danceability,
             energy,
             popularity
        )
        SELECT DISTINCT id, name, artists, duration_ms, lyrics, genre, niche_genres, loudness, speechiness, instrumentalness, liveness, valence, tempo, key, mode, acousticness, danceability, energy, popularity
        FROM (
                SELECT *, ROW_NUMBER() OVER (
                        PARTITION BY name, artists, lyrics
                        ORDER BY id
                ) as row_num
                FROM songs_raw
        )
        WHERE row_num = 1;
        """)
print('Num. rows before removing duplicates:', con.execute("SELECT COUNT(*) FROM songs_raw").fetchall())
print('Num. rows after removing duplicates:', con.execute("SELECT COUNT(*) FROM songs_no_duplicates").fetchall())

Num. rows before removing duplicates: [(550622,)]
Num. rows after removing duplicates: [(498978,)]


Check to see if a specific artist is included

In [8]:
con.sql("""
    SELECT *
    FROM songs_no_duplicates
    WHERE artists_list LIKE '%ANOTR%'
""").show()

┌────────────────────────┬───────────────┬──────────────────────────┬─────────────┬─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## Step 2: Request Deezer API for Additional Information

### **Step 2a:** Setup

In [9]:
con.execute('DROP TABLE IF EXISTS deezer_data')
con.execute("""
    CREATE TABLE deezer_data (
        _id VARCHAR,
        deezer_id INT64,
        name VARCHAR,
        release_date VARCHAR,
        explicit BOOLEAN,
        deezer_rank INTEGER,
        PRIMARY KEY(_id)
    )
""")

In [10]:
def get_song_details(artist_name, song_name):
    #Performs a fuzzy search on the artist and song name
    search_query = f"{artist_name} {song_name}"
    search_url = f"https://api.deezer.com/search?q={search_query}"
    
    try:
        search_res = requests.get(search_url).json()
        
        if search_res.get('total', 0) > 0:
            #Take the first result
            track_data = search_res['data'][0]
            
            track_id = track_data['id']
            is_explicit = track_data.get('explicit_lyrics')
            album_id = track_data.get('album', {}).get('id')
            
            #Get the album details for the release date
            album_url = f"https://api.deezer.com/album/{album_id}"
            album_res = requests.get(album_url).json()
            
            release_date = album_res.get('release_date')
            
            return {
                "track_id": track_id,
                "explicit": is_explicit,
                "release_date": release_date,
                "deezer_rank": track_data.get('rank')
            }
            
    except Exception as e:
        print(f"Error fetching {song_name}: {e}")
        
    return None

### **Step 2b:** Make list of songs to be requested

In [11]:
songs_to_request = con.execute("""
    SELECT _id, song_name, artists_list
    FROM songs_no_duplicates
""").fetchall()
print(len(songs_to_request))

ids_to_request = [id for id, _, _ in songs_to_request]
print(len(ids_to_request))

498978
498978


### **Step 2c:** Run API query

> **⚠️ Warning:** This takes several days to run across all 498,978 songs. If the notebook shows a `KeyboardInterrupt` exception below, that is because the code was briefly run again before submission for testing purposes, but was interrupted due to time constraints. As above, this code can be used to test/grade our API requesting capabilities, but will take several days to fully execute and build a new database. See the `data/song_data.db` file for a fully-built database.

In [12]:
ids_already_done = con.execute("""
SELECT _id FROM deezer_data
""").fetchall()
successful_count = len(ids_already_done)
total_count = con.execute("""
    SELECT COUNT(*)
    FROM songs_no_duplicates
""")
start_time = datetime.now()

for row in songs_to_request:
    row_start_time = datetime.now()
    id = row[0]
    song_name = row[1]
    artists_raw = row[2]
    artists = artists_raw.replace('[', '').replace(']', '').replace(',', '').replace('"', '')

    if id in ids_to_request:
        print('Requesting song id', id)
        details = get_song_details(artists, song_name)
        if details == None: #if details is None:
            print('Details returned None for song id', id)
        else:
            entry = (
                id,
                details['track_id'],
                song_name,
                details['release_date'],
                details['explicit'],
                details['deezer_rank']
            )

            try:
                con.execute("""
                    INSERT INTO deezer_data (_id, deezer_id, name, release_date, explicit, deezer_rank)
                    VALUES (?, ?, ?, ?, ?, ?)
                """, entry)

                successful_count += 1
                print(f'Successfully logged song id {id} ({successful_count}/{total_count})')
                print('Duration of', (datetime.now() - row_start_time))
                ids_to_request.remove(id)
                ids_already_done.append(id)
            
            except duckdb.ConstraintException as ce:
                print(f'ConstraintError (may already have data for song id {id}): {ce}')

duration = datetime.now() - start_time
print(f'Took {duration} to complete = avg. of {duration/total_count} per song')
print(
'Total number of songs with deezer data:', con.execute("""
    SELECT COUNT(*)
    FROM deezer_data
""").fetchall()
)

Requesting song id 2v0AVgNMYwDs6kQUvBESRm
Successfully logged song id 2v0AVgNMYwDs6kQUvBESRm (1/<_duckdb.DuckDBPyConnection object at 0x10cee80b0>)
Duration of 0:00:00.622031
Requesting song id 2xCFsgvXuIYgRoXy6ZhBIy
Details returned None for song id 2xCFsgvXuIYgRoXy6ZhBIy
Requesting song id 3sihMPjivvbqhvyQfm0VJf
Successfully logged song id 3sihMPjivvbqhvyQfm0VJf (2/<_duckdb.DuckDBPyConnection object at 0x10cee80b0>)
Duration of 0:00:00.576914
Requesting song id 5jKZsuUzvV5bjdCRMn73rD
Successfully logged song id 5jKZsuUzvV5bjdCRMn73rD (3/<_duckdb.DuckDBPyConnection object at 0x10cee80b0>)
Duration of 0:00:00.865836
Requesting song id 3LGqAOlRRqixUDALNlPWWH
Successfully logged song id 3LGqAOlRRqixUDALNlPWWH (4/<_duckdb.DuckDBPyConnection object at 0x10cee80b0>)
Duration of 0:00:00.657194
Requesting song id 6MpO4Pt68ufyhKGryLzlap
Successfully logged song id 6MpO4Pt68ufyhKGryLzlap (5/<_duckdb.DuckDBPyConnection object at 0x10cee80b0>)
Duration of 0:00:00.730825
Requesting song id 7hfeFlE

KeyboardInterrupt: 

In [13]:
print('Length of deezer_data table:', con.execute(
    """SELECT COUNT(*)
    FROM deezer_data"""
    ).fetchall()
)

Length of deezer_data table: [(49,)]


## **Step 3:** Merging songs_no_duplicates and deezer_data tables

### **Step 3a:** Setup
Create songs table to hold final data.

In [14]:
con.execute('DROP TABLE IF EXISTS songs')
con.execute("""
    CREATE TABLE songs (
        _id VARCHAR,
        song_name VARCHAR,
        artists_list VARCHAR,
        duration_ms BIGINT,
        lyrics VARCHAR,
        genre VARCHAR,
        niche_genres VARCHAR,
        loudness DOUBLE,
        speechiness DOUBLE,
        instrumentalness DOUBLE,
        liveness DOUBLE,
        valence DOUBLE,
        tempo DOUBLE,
        key BIGINT,
        mode BIGINT,
        acousticness DOUBLE,
        danceability DOUBLE,
        energy DOUBLE,
        popularity BIGINT,
        deezer_id INT64,
        release_date VARCHAR,
        explicit BOOLEAN,
        deezer_rank INTEGER,
        PRIMARY KEY(_id)
    )
""")

### **Step 3b:** Performing the Inner Join
Finally, we must perform an inner join on the songs_no_duplicates and deezer_data tables.

In [15]:
start_time = datetime.now()
print('Length of songs_no_duplicates table:', con.execute(
    """SELECT COUNT(*)
    FROM songs_no_duplicates"""
    ).fetchall()
)
print('Length of deezer_data table:', con.execute(
    """SELECT COUNT(*)
    FROM deezer_data"""
    ).fetchall()
)
con.execute("""
    INSERT INTO songs
    SELECT
        s._id,
        s.song_name,
        s.artists_list,
        s.duration_ms,
        s.lyrics,
        s.genre,
        s.niche_genres,
        s.loudness,
        s.speechiness,
        s.instrumentalness,
        s.liveness,
        s.valence,
        s.tempo,
        s.key,
        s.mode,
        s.acousticness,
        s.danceability,
        s.energy,
        s.popularity,
        dd.deezer_id,
        dd.release_date,
        dd.explicit,
        dd.deezer_rank
    FROM deezer_data dd
    INNER JOIN songs_no_duplicates s
    ON s._id = dd._id
""")

print('Length of resulting table:', con.execute(
    """SELECT COUNT(*)
    FROM songs"""
    ).fetchall()
)
print('Duration:', datetime.now() - start_time)
con.table('songs').limit(5).show()

Length of songs_no_duplicates table: [(498978,)]
Length of deezer_data table: [(49,)]
Length of resulting table: [(49,)]
Duration: 0:00:00.042812
┌────────────────────────┬──────────────────────────────────────────────┬──────────────────────┬─────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

## **Step 4:** Final Steps
Now that we are done with our data collection, we can close the connection, leaving us with a songs.db file containing a songs table, ready for the next step of our project.

### **Step 4a:** Drop Intermediate Tables
We no longer need the songs_raw, songs_no_duplicates, and deezer_data tables, so to save space, we will drop them below.

In [16]:
con.execute('DROP TABLE IF EXISTS songs_raw')
con.execute('DROP TABLE IF EXISTS songs_no_duplicates')
con.execute('DROP TABLE IF EXISTS deezer_data')

### **Step 4b:** Close DuckDB Connection
This will allow us to take our final songs.db file and pass it on to further stages of the project.

In [17]:
con.close()

## **Step 5:** The End
Please see **README.md** for instructions on running the rest of the project.